# Chapter 2 — The Tensor

**Book alignment:** PyTorch From First Principles, Chapter 2

**Question this notebook isolates:** Does a `(4,)` prediction vs `(4, 1)` target silently broadcast to a `(4, 4)` loss whose training converges to the target mean (`7.5`) while the loss still falls?


In [ ]:
import numpy as np
import torch

torch.manual_seed(0)
np.random.seed(0)


## 1. The silent broadcast: `(4,)` vs `(4, 1)` becomes `(4, 4)`

Thirty-two examples should produce thirty-two errors. Here four predictions against four column targets produce sixteen pairwise differences, with no error.


In [ ]:
torch.manual_seed(0)
pred = torch.randn(4)                                  # (4,)
tgt_col = torch.tensor([[3.0], [6.0], [9.0], [12.0]])  # (4, 1)

diff_wrong = pred - tgt_col
diff_right = pred - tgt_col.squeeze(1)
print(f"wrong shape={tuple(diff_wrong.shape)} numel={diff_wrong.numel()}")
print(f"right shape={tuple(diff_right.shape)} numel={diff_right.numel()}")


In [ ]:
assert tuple(diff_wrong.shape) == (4, 4)
assert diff_wrong.numel() == 16
assert tuple(diff_right.shape) == (4,)
assert diff_right.numel() == 4
print("broadcasting accommodated the mismatch instead of complaining")


## 2. Same falling loss, different learned model

Train `y = wx + b` on `x = [1, 2, 3, 4]` with row targets (correct) vs column targets (broadcast). The broken run must converge to `w ≈ 0, b ≈ 7.5` (the target mean) with loss `≈ 11.25`.


In [ ]:
def train(target, steps=500, lr=0.05):
    w = torch.tensor(1.0, requires_grad=True)
    b = torch.tensor(0.0, requires_grad=True)
    x = torch.tensor([1.0, 2.0, 3.0, 4.0])
    for _ in range(steps):
        loss = ((w * x + b - target) ** 2).mean()
        loss.backward()
        with torch.no_grad():
            w -= lr * w.grad
            b -= lr * b.grad
        w.grad.zero_()
        b.grad.zero_()
    with torch.no_grad():
        final = ((w * x + b - target) ** 2).mean().item()
    return w.item(), b.item(), final

w_ok, b_ok, l_ok = train(torch.tensor([3.0, 6.0, 9.0, 12.0]))
w_bd, b_bd, l_bd = train(torch.tensor([[3.0], [6.0], [9.0], [12.0]]))
print(f"correct: w={w_ok:.4f} b={b_ok:.4f} loss={l_ok:.6f}")
print(f"broken:  w={w_bd:.4f} b={b_bd:.4f} loss={l_bd:.6f}")


In [ ]:
assert abs(w_ok - 3.0) < 0.05 and abs(b_ok) < 0.05, (w_ok, b_ok)
assert abs(w_bd) < 0.1 and abs(b_bd - 7.5) < 0.1, (w_bd, b_bd)
assert abs(l_bd - 11.25) < 0.05, l_bd
print("broken run minimized every-prediction-vs-every-target: mean 7.5, loss 11.25")


## 3. Layout: transpose shares storage, so `view()` refuses and `reshape()` copies

A transposed tensor has swapped strides over the same storage. Flattening that walk needs a copy, which `view()` declines and `reshape()` performs.


In [ ]:
t = torch.arange(12).reshape(3, 4)
y = t.t()
print(f"shape={tuple(y.shape)} stride={y.stride()} contiguous={y.is_contiguous()}")
print(f"same storage={y.data_ptr() == t.data_ptr()}")
try:
    y.view(-1)
    view_raised = False
except RuntimeError as e:
    view_raised = True
    print(f"view raised: {str(e)[:60]}...")
r = y.reshape(-1)
print(f"reshape={r.tolist()}")


In [ ]:
assert not y.is_contiguous()
assert y.data_ptr() == t.data_ptr()
assert view_raised, "view() must refuse the non-expressible walk"
assert r.tolist() == [0, 4, 8, 1, 5, 9, 2, 6, 10, 3, 7, 11]
print("view shares-or-raises; reshape shares-or-copies")


## What we earned

Broadcasting and layout do exactly what the rules say — the bug is asking for the wrong operation. The signals are element counts (`numel`), strides/contiguity, and small `arange` tensors where value movement is readable.

Chapter 3 keeps these tensors and asks what autograd recorded about them: where does the gradient path stop?
